# Module 7.4 — Bytecode & AST Analysis

### Python Mastery · Track 7: Performance, Internals & C Extensions

Python source is compiled to **bytecode**, a low-level instruction set that the interpreter executes. Looking at bytecode reveals what your code really does, and the **abstract syntax tree** (AST) gives a structured view of code that tools can analyse and transform. This module covers both, building on the AST introduction from Module 4.6.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Everything uses the standard library (`dis`, `ast`) and runs directly in cells.
- Bytecode details vary between Python versions; the concepts are stable even when specific instruction names differ. This notebook runs on Python 3.12.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain the compile-to-bytecode pipeline.
2. Inspect bytecode with `dis` and read the disassembly.
3. Compare implementations at the bytecode level.
4. Parse code to an AST and walk it for analysis.
5. Build a small AST-based linter that flags a pattern.

**Prerequisites:** Tracks 1 to 6 (especially Module 4.6), and Modules 7.1 to 7.3.

---

## Part 1 · From Source to Bytecode

Running Python source happens in stages: the source is **parsed** into an abstract syntax tree, the tree is **compiled** into bytecode (stored in a code object), and the interpreter's evaluation loop **executes** that bytecode. The bytecode lives on the code object as the `co_code` attribute, with related metadata such as constants and variable names alongside it.

In [ ]:
def example(x):
    y = x + 1
    return y * 2

# The compiled bytecode and its metadata live on the function's code object.
code_obj = example.__code__
print("constants used:", code_obj.co_consts)
print("local variable names:", code_obj.co_varnames)
print("argument count:", code_obj.co_argcount)
print("raw bytecode is bytes:", isinstance(code_obj.co_code, bytes), f"({len(code_obj.co_code)} bytes)")

## Part 2 · Disassembling with `dis`

Raw bytecode is unreadable as bytes, so the `dis` module disassembles it into named instructions. Each line shows an operation such as `LOAD_FAST` (push a local onto the stack), `BINARY_OP` (apply an operator), or `RETURN_VALUE`. Reading this tells you exactly what steps the interpreter performs.

In [ ]:
import dis

def add_one(x):
    return x + 1

# Disassemble the function into human-readable instructions.
dis.dis(add_one)

The interpreter is a **stack machine**: most instructions push values onto or pop them off an internal stack. For `x + 1`, it loads `x`, loads the constant `1`, applies the addition (which pops both and pushes the result), then returns the top of the stack. Seeing this makes the execution model concrete.

In [ ]:
import dis

# A slightly larger function shows control flow in the bytecode.
def classify(n):
    if n > 0:
        return "positive"
    return "non-positive"

dis.dis(classify)
# Note the comparison, the conditional jump, and the two return paths.

## Part 3 · Comparing Implementations at the Bytecode Level

Bytecode can explain *why* one version is faster than another. Comparing the disassembly of two equivalent functions shows differences in the work each performs. Here we compare a list built by repeated `.append` in a loop against a list comprehension; the comprehension uses a specialised, tighter instruction sequence.

In [ ]:
import dis

def build_loop():
    result = []
    for i in range(5):
        result.append(i)
    return result

def build_comprehension():
    return [i for i in range(5)]

print("=== loop with append ===")
dis.dis(build_loop)
print("=== list comprehension ===")
dis.dis(build_comprehension)

The loop version repeatedly loads the `append` method and calls it, instruction by instruction, while the comprehension uses a dedicated `LIST_APPEND` step inside a compact loop, with less per-iteration overhead. This is the bytecode-level reason comprehensions tend to be faster than equivalent explicit loops, complementing the timing evidence from Module 7.1.

## Part 4 · From Bytecode Back Up to the AST

Bytecode is the low-level view; the **abstract syntax tree** is the high-level structured view, sitting between source and bytecode. Module 4.6 introduced parsing with `ast.parse` and walking with `ast.NodeVisitor`. Here we use the AST for analysis: examining structure without executing code. `ast.dump` shows the tree, and `ast.walk` visits every node.

In [ ]:
import ast

source = "result = (a + b) * count"
tree = ast.parse(source, mode="exec")

# A compact view of the tree's structure.
print(ast.dump(tree, indent=2)[:300], "...")
print()

# Walk every node and tally the kinds present.
from collections import Counter
kinds = Counter(type(node).__name__ for node in ast.walk(tree))
print("node kinds in this snippet:", dict(kinds))

## Part 5 · A Small AST-Based Linter

Because the AST is structured data, you can write tools that detect patterns in code, exactly how real linters (Module 6.4) work internally. The example below builds a tiny linter that flags two questionable patterns: a bare `except:` clause (which catches everything, including system exits) and use of `eval` (a security risk, per Module 6.7). It reports findings without running the code.

In [ ]:
import ast

class SimpleLinter(ast.NodeVisitor):
    """Flag bare excepts and calls to eval."""
    def __init__(self):
        self.warnings = []

    def visit_ExceptHandler(self, node):
        if node.type is None:                       # 'except:' with no exception type
            self.warnings.append(f"line {node.lineno}: bare 'except:' catches everything")
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id == "eval":
            self.warnings.append(f"line {node.lineno}: use of 'eval' is risky")
        self.generic_visit(node)

source = """
def risky(data):
    try:
        return eval(data)
    except:
        return None
"""

linter = SimpleLinter()
linter.visit(ast.parse(source))
print("linter findings:")
for warning in linter.warnings:
    print("  -", warning)

This little tool embodies the principle behind static analysis: parse code to an AST, walk the tree looking for patterns, and report. Real linters extend this to hundreds of rules, but the mechanism is the same one shown here.

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Disassembling a simple function (Easy)

In [ ]:
import dis

def square(x):
    return x * x

dis.dis(square)

### Example 2 — Inspecting a code object's metadata (Easy)

In [ ]:
def greet(name, greeting="hi"):
    message = greeting + " " + name
    return message

co = greet.__code__
print("constants:", co.co_consts)
print("locals:", co.co_varnames)
print("arg count:", co.co_argcount)

### Example 3 — Comparing two ways to sum (Medium)

In [ ]:
import dis

def sum_loop():
    total = 0
    for i in range(3):
        total += i
    return total

def sum_builtin():
    return sum(range(3))

print("=== explicit loop ===")
dis.dis(sum_loop)
print("=== built-in sum ===")
dis.dis(sum_builtin)
print("the built-in does the looping in C, hence the much shorter bytecode")

### Example 4 — Counting node types in code (Medium)

In [ ]:
import ast
from collections import Counter

source = """
def f(x):
    if x > 0:
        return x + 1
    return x - 1
"""
tree = ast.parse(source)
counts = Counter(type(n).__name__ for n in ast.walk(tree))
for kind, n in sorted(counts.items()):
    print(f"{kind}: {n}")

### Example 5 — A linter that flags mutable default arguments (Difficult)

In [ ]:
import ast

class MutableDefaultLinter(ast.NodeVisitor):
    """Flag functions with a mutable default argument (a classic bug, see Module 1.4)."""
    def __init__(self):
        self.warnings = []

    def visit_FunctionDef(self, node):
        for default in node.args.defaults:
            if isinstance(default, (ast.List, ast.Dict, ast.Set)):
                self.warnings.append(
                    f"line {node.lineno}: function '{node.name}' has a mutable default argument"
                )
        self.generic_visit(node)

source = """
def good(items=None):
    items = items or []
    return items

def bad(items=[]):
    items.append(1)
    return items
"""
linter = MutableDefaultLinter()
linter.visit(ast.parse(source))
print("findings:", linter.warnings)

### Example 6 — Counting function calls via the AST (Difficult)

In [ ]:
import ast

class CallCounter(ast.NodeVisitor):
    def __init__(self):
        self.calls = {}
    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            name = node.func.id
            self.calls[name] = self.calls.get(name, 0) + 1
        self.generic_visit(node)

source = """
print(len(data))
print(max(data))
print(min(data))
total = sum(data)
"""
counter = CallCounter()
counter.visit(ast.parse(source))
print("call counts:", counter.calls)

---

## Exercises

**Exercise 1 (Easy).** Use `dis.dis` to disassemble a function that returns `x + y`, and identify the instruction that performs the addition in the output.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Define a function with one constant and two locals, and print its code object's `co_consts` and `co_varnames`.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Disassemble both a list comprehension function and an equivalent append-loop function, and note (in a comment) which uses fewer instructions per element.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Parse a small snippet to an AST and use `ast.walk` with a `Counter` to report how many `Name`, `BinOp`, and `Call` nodes it contains.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Write an `ast.NodeVisitor` linter that flags any use of `print` (as if enforcing a "use logging instead" rule), reporting the line number of each occurrence. Test it on a snippet with two prints.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import dis
def add(x, y):
    return x + y
dis.dis(add)
print("the BINARY_OP instruction performs the addition")

**Solution 2**

In [ ]:
def f(a, b):
    c = a + b + 10
    return c
co = f.__code__
print("consts:", co.co_consts)
print("varnames:", co.co_varnames)

**Solution 3**

In [ ]:
import dis
def comp(): return [i for i in range(4)]
def loop():
    r = []
    for i in range(4):
        r.append(i)
    return r
print("=== comprehension ===")
dis.dis(comp)
print("=== loop ===")
dis.dis(loop)
print("the comprehension uses fewer instructions per element (LIST_APPEND vs method call)")

**Solution 4**

In [ ]:
import ast
from collections import Counter
source = "y = f(a + b) + g(c)"
counts = Counter(type(n).__name__ for n in ast.walk(ast.parse(source)))
for kind in ["Name", "BinOp", "Call"]:
    print(kind, counts.get(kind, 0))

**Solution 5**

In [ ]:
import ast

class NoPrintLinter(ast.NodeVisitor):
    def __init__(self):
        self.hits = []
    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id == "print":
            self.hits.append(node.lineno)
        self.generic_visit(node)

source = """
print("one")
x = 5
print("two")
"""
linter = NoPrintLinter()
linter.visit(ast.parse(source))
print("print used on lines:", linter.hits)

---

## Summary and Key Points

- Running Python proceeds in stages: source is parsed to an AST, compiled to bytecode stored on a code object, and executed by the interpreter loop; the code object also holds constants, variable names, and argument counts.
- `dis.dis` disassembles bytecode into named instructions; the interpreter is a stack machine, loading values, applying operations that pop and push, and returning the top of the stack.
- Comparing disassembly explains performance differences: a list comprehension uses a compact `LIST_APPEND` loop, while an append-loop repeatedly loads and calls the method, which is why comprehensions are typically faster.
- The AST is the high-level structured view between source and bytecode; `ast.parse` builds it, `ast.dump` shows it, and `ast.walk` visits every node for analysis.
- An `ast.NodeVisitor` that walks the tree looking for patterns is exactly how linters work; small visitors can flag bare excepts, `eval`, mutable default arguments, and more, without running the code.

### Next module: 7.5 — Interfacing with C

The next module connects Python to C: calling shared libraries with `ctypes` and CFFI, accelerating code with Cython, and building a minimal CPython C extension.